In [9]:
import os
import json
import psycopg2
import psycopg2.extras
from openai import OpenAI
from pgvector.psycopg2 import register_vector
from pathlib import Path
from dotenv import load_dotenv
root = Path.cwd()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Start the notebook from inside the cloned project.")
    root = root.parent

load_dotenv(root / "prod.env")

neon_connection_string = os.environ.get("NEON_DATABASE_URL")
openai_api_key = os.environ.get("OPENAI_KEY")

INPUT_DIR = root / "src" / "publications"
client = OpenAI(api_key=openai_api_key)
MODEL = "text-embedding-3-small"

In [ ]:
def get_embedding(text):
    '''Get the embedding for a given text using the OpenAI API.'''
    response = client.embeddings.create(
        model=MODEL,
        input=text,
        dimensions=512
    )
    return response.data[0].embedding

In [11]:
def search_publications(prompt, top_k=5):
    '''Search for publications related to the given prompt using OpenAI embeddings.
    https://help.openai.com/en/articles/6824809-embeddings-faq'''
    embedding = get_embedding(prompt)
    conn = psycopg2.connect(neon_connection_string)
    register_vector(conn.cursor())
    cursor = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
    cursor.execute("""
        SELECT  id, 
                source, 
                source_id, 
                publication_date, 
                doi, 
                url, 
                title, 
                abstract, 
                authors,
                embedding <=> %s::vector AS distance
        FROM publications
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (embedding, embedding, top_k))
    results = cursor.fetchall()
    conn.close()
    return results

In [12]:
prompt = "Find Publications related to Superior Colorectal Polyp Detection"
results = search_publications(prompt)

In [13]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame([dict(result) for result in results])

with pd.option_context("display.max_colwidth", 120):
    display(df)

,id,source,source_id,publication_date,doi,url,title,abstract,authors,distance
0,18889,semantic_scholar,01c5bf9ecaa0f2dc42ee4b10c5b9ac4b90cb4797,2025-03-03,10.3390/gastroent16010009,https://www.semanticscholar.org/paper/01c5bf9ecaa0f2dc42ee4b10c5b9ac4b90cb4797,Strategies to Enhance the Adenoma Detection Rate (ADR) and the Serrated Polyp Detection Rate (SPDR) in Colonoscopy: ...,"Introduction: High-quality colonoscopy is influenced by several factors, with the adenoma detection rate (ADR) being...","{""Davide Scalvini"",""S. Agazzi"",""S. Maimaris"",""L. Rovedatti"",""Daniele Brinch"",""A. Cappellini"",""C. Ciccioli"",""Michele ...",0.402520
1,12609,semantic_scholar,350ff271419213cb8967738d82e62e6cb71256b4,2025-10-02,10.1055/a-2706-0201,https://www.semanticscholar.org/paper/350ff271419213cb8967738d82e62e6cb71256b4,Artificial intelligence for colorectal polyp sizing: clinical promise and methodological challenges,NaN,"{""D. von Renteln""}",0.411132
2,2664,semantic_scholar,1632fe57ac35ab8086b69857647695f1b9cff91c,2020-04-01,10.1016/j.tgie.2019.150631,https://www.semanticscholar.org/paper/1632fe57ac35ab8086b69857647695f1b9cff91c,Artificial intelligence for colon polyp detection: Why should we embrace this?,Abstract Optimal success of colonoscopy for prevention of colorectal cancer is currently measured by adenoma detecti...,"{""Robin Zachariah"",""Andrew Q. Ninh"",""William E. Karnes""}",0.432062
3,9400,semantic_scholar,23178324148ff16fa70fb306a42040b6bfd4a1ee,2025-02-26,10.1038/s41598-025-91467-1,https://www.semanticscholar.org/paper/23178324148ff16fa70fb306a42040b6bfd4a1ee,A colonic polyps detection algorithm based on an improved YOLOv5s,"Colon cancer is a prevalent malignancy, substantially it prevented most effectively from killing patients through ea...","{""Jianjun Li"",""Jinhui Zhao"",""Yifan Wang"",""Jin-Hui Zhu"",""Yanhong Wei"",""Jun-Jiang Zhu"",""Xiaolu Li"",""Shu-Bin Yan"",""Qich...",0.440379
4,15245,semantic_scholar,2cdb85e07ac80e43d7b1a520c7204c19c49b36cb,2023-05-23,10.9734/jammr/2023/v35i145060,https://www.semanticscholar.org/paper/2cdb85e07ac80e43d7b1a520c7204c19c49b36cb,The Efficacy of AI-Assisted CADe Systems in Colorectal Polyps and Adenomas during Colonoscopies: A Systematic Review,Colorectal polyps and adenomas are a significant health concern that can lead to colorectal cancer. Detecting polyps...,"{""Stella Nwosu"",""Norense Ehimwenma"",""Victor N. Oboli"",""Faiza Arslan"",""Khizar Khan"",""Akata Abung"",""Aqsa Latif"",""Danie...",0.450029


In [14]:
# Summarize Database content, quantity of total publications per source
def summarize_publications_by_source():
    conn = psycopg2.connect(neon_connection_string)
    cursor = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
    cursor.execute("""
        SELECT source, COUNT(source_id) AS total_publications
        FROM publications
        GROUP BY source
        ORDER BY total_publications DESC
    """)
    results = cursor.fetchall()
    conn.close()
    return results

db_summary = summarize_publications_by_source()
for row in db_summary:
    print(f"Source: {row['source']}, Total Publications: {row['total_publications']}")

Source: semantic_scholar, Total Publications: 8951
Source: openalex, Total Publications: 7949
Source: arxiv, Total Publications: 1087
